In [ ]:
import numpy as np
import torch

from si.detection import get_j_in_topk_intervals
from si.dnn.dnn import get_model_intervals
from si.util import (build_initial_sign_interval, build_test_reference_contrast_path,
    gen_data, load_models, load_working_model_artifacts, truncated_cdf,)

In [12]:
d = 10             # feature dimension
n_s = 250          # n_s
n_t = 50           # n_t
n_ref = 200        # n_ref
top_k = 0.05    # top 5% as anomalies

X_source, _ = gen_data(mu = 1.0, delta = 0.0, n = n_s, 
                       d=d, anomaly_rate=0.0, rho=0.0, seed=0)
X_target, _ = gen_data(mu=0.0, delta=0.0, n=n_t, d=d, 
                       anomaly_rate=0.0, rho=0.0, seed=1)
X_ref, _ = gen_data(mu=0.0, delta=0.0, n=n_ref, d=d, anomaly_rate=0.0, rho=0.0, seed=2)

print("X_source:", X_source.shape)
print("X_target:", X_target.shape)
print("X_ref:   ", X_ref.shape)

X_source: (250, 10)
X_target: (50, 10)
X_ref:    (200, 10)


In [13]:
# rho = 0.0, use correlated for rho = 0.5
model_name = "deepsad_da_delta2_independent" 

encoder, deepsad_c, device = load_models(device="cpu", model_dir="models", model_name=model_name,
    d=d, h_dims=[64, 32], rep_dim=8,
)
deepsad_c = np.asarray(deepsad_c, dtype=np.float64)

artifacts = load_working_model_artifacts(model_dir="models", model_name= model_name)

Sigma_s = artifacts["Sigma_source"]
Sigma_t = artifacts["Sigma_target"]

print("Network: ", type(encoder).__name__)
print("Center c:", deepsad_c.shape)
print("Sigma^s: ", Sigma_s.shape, "| Sigma^t:", Sigma_t.shape)

Network:  DADeepSADEncoder
Center c: (8,)
Sigma^s:  (10, 10) | Sigma^t: (10, 10)


In [24]:
X = np.vstack([X_source, X_target])

with torch.no_grad():
    embeddings = encoder(torch.tensor(X, dtype=torch.float32)).numpy()
scores = np.linalg.norm(embeddings - deepsad_c, axis=1)

k = max(1, int(top_k * len(scores)))
O = sorted(np.argpartition(scores, -k)[-k:])

O_t = [int(i) for i in O if int(i) >= n_s]
if not O_t:
    raise RuntimeError("No target instance detected")

j_global = O_t[0]
j_target = j_global - n_s

print("Anomalies:", O_t)

Anomalies: [265, 266, 275, 289]


In [22]:
path = build_test_reference_contrast_path(X_test = X_target, X_ref = X_ref, j = j_target,
    X_source = X_source, Sigma_source = Sigma_s, Sigma_target = Sigma_t)

print(f"observed T_j = {path['test_statistic']:.6f}")
print(f"sigma = {path['sigma']:.6f}")

left, right = build_initial_sign_interval(
    path["a_test"], path["b_test"], j_target, path["positive_sign"],
    path["reference_a_mean"], path["reference_b_mean"], path["sigma"],
)
print(f"Z_2 = [{left:.4f}, {right:.4f}]")

a = np.vstack([path["a_source"], path["a_test"]])
b = np.vstack([path["b_source"], path["b_test"]])

intervals = get_model_intervals(encoder, [(left, right, a, b)])
print(f"sub-intervals after the line search: {len(intervals)}")

intervals = get_j_in_topk_intervals(
    intervals, top_k_percent= top_k, deepsad_c=deepsad_c, j=j_global
)

truncation = [(lo, hi, bool(in_O)) for lo, hi, in_O in intervals]
print(f"pieces: {len(truncation)} total, {sum(1 for *_, k in truncation if k)} with j in O")

observed T_j = 7.601378
sigma = 3.170173
Z_2 = [7.4914, 63.4035]
sub-intervals after the line search: 46
pieces: 74 total, 74 with j in O


In [23]:
cdf = truncated_cdf(0.0, path["sigma"], truncation, True, path["test_statistic"])
p_value = 2 * min(cdf, 1 - cdf)


print(f"selective p-value = {p_value:.6f}")

selective p-value = 0.179790
